## Performance Describe

绩效指标的描述性统计，包括聚合到被试维度的平均值分布，以及按物品维度来统计平均值分布。

In [ ]:
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

sns.set_style("white")
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
performance_dir = Path('../../data/analysis/performance')
performance_file = performance_dir / 'performance.csv'

performance_df = pd.read_csv(performance_file)

## 被试维度

In [ ]:
participant_performance = performance_df.groupby('participant_id').agg({
    'fluency': 'sum',
    'originality': 'mean',
}).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(data=participant_performance, x='fluency', kde=True, ax=axes[0], color='skyblue', edgecolor='black', binwidth=5)
axes[0].xaxis.set_major_locator(ticker.MultipleLocator(5))
axes[0].set_title('流畅性分布', fontsize=16)
axes[0].set_xlabel('流畅性', fontsize=14)
axes[0].set_ylabel('频数', fontsize=14)
axes[0].grid(axis='y', linestyle='--', alpha=0.7)
axes[0].set_axisbelow(True)

originality_data = participant_performance['originality'].dropna()
sns.histplot(data=originality_data, kde=True, ax=axes[1], color='salmon', edgecolor='black', binwidth=0.1)
axes[1].xaxis.set_major_locator(ticker.MultipleLocator(0.1))
axes[1].set_title('原创性分布', fontsize=16)
axes[1].set_xlabel('原创性', fontsize=14)
axes[1].set_ylabel('频数', fontsize=14)
axes[1].grid(axis='y', linestyle='--', alpha=0.7)
axes[1].set_axisbelow(True)

sns.despine()
plt.tight_layout()
plt.show()

## 试次维度

In [ ]:
trial_performance = performance_df.groupby(['participant_id', 'trial_index']).agg({
    'fluency': 'sum',
    'originality': 'mean',
}).reset_index()

display(trial_performance.describe())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(data=trial_performance, x='fluency', kde=True, ax=axes[0], color='skyblue', edgecolor='black', binwidth=1)
axes[0].xaxis.set_major_locator(ticker.MultipleLocator(1))
axes[0].set_title('流畅性分布', fontsize=16)
axes[0].set_xlabel('流畅性', fontsize=14)
axes[0].set_ylabel('试次数', fontsize=14)
axes[0].grid(axis='y', linestyle='--', alpha=0.7)
axes[0].set_axisbelow(True)

originality_data = trial_performance['originality'].dropna()
sns.histplot(data=originality_data, kde=True, ax=axes[1], color='salmon', edgecolor='black', binwidth=0.1)
axes[1].xaxis.set_major_locator(ticker.MultipleLocator(0.1))
axes[1].set_title('原创性分布', fontsize=16)
axes[1].set_xlabel('原创性', fontsize=14)
axes[1].set_ylabel('试次数', fontsize=14)
axes[1].grid(axis='y', linestyle='--', alpha=0.7)
axes[1].set_axisbelow(True)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# 高质量回答数：above_median

trial_above = performance_df.groupby(['participant_id', 'trial_index']).agg({
    'above_median': 'sum',
    'above_median_ratio': 'mean',
}).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(data=trial_above, x='above_median', kde=True, ax=axes[0], color='mediumseagreen', edgecolor='black', binwidth=1)
axes[0].xaxis.set_major_locator(ticker.MultipleLocator(1))
axes[0].set_title('高质量回答数分布', fontsize=16)
axes[0].set_xlabel('高质量回答数', fontsize=14)
axes[0].set_ylabel('试次数', fontsize=14)
axes[0].grid(axis='y', linestyle='--', alpha=0.7)
axes[0].set_axisbelow(True)

sns.histplot(data=trial_above, x='above_median_ratio', kde=True, ax=axes[1], color='mediumseagreen', edgecolor='black', binwidth=0.05)
axes[1].xaxis.set_major_locator(ticker.MultipleLocator(0.1))
axes[1].set_title('高质量回答占比分布', fontsize=16)
axes[1].set_xlabel('高质量回答占比', fontsize=14)
axes[1].set_ylabel('试次数', fontsize=14)
axes[1].grid(axis='y', linestyle='--', alpha=0.7)
axes[1].set_axisbelow(True)

sns.despine()
plt.tight_layout()
plt.show()

## 物品维度

In [ ]:
desired_item_order = [
    '罐头', '气球', '勺子', '雨伞',
    '钥匙', '绳子', '砖头', '刀',
    '衣架', '盒子', '螺丝刀', '袜子',
    '床单', '报纸', '轮胎', '鞋子',
]
available_items = set(performance_df['item_name'].dropna().unique())
item_names = [item for item in desired_item_order if item in available_items]

if len(item_names) != 16:
    print(f'⚠警告：当前检测到 {len(item_names)} 个物品，不是 16 个。图会按实际物品数量绘制。')

item_summaries = []
for item_name in item_names:
    item_df = performance_df[performance_df['item_name'] == item_name]
    item_performance = item_df.groupby('participant_id').agg({
        'fluency': 'sum',
        'originality': 'mean',
    }).reset_index()
    item_summaries.append(item_performance)

all_item_performance = pd.concat(item_summaries, ignore_index=True)
fluency_xlim = (0, int(all_item_performance['fluency'].max()) + 1)
originality_min = all_item_performance['originality'].min()
originality_max = all_item_performance['originality'].max()
originality_xlim = (round(originality_min - 0.1, 1), round(originality_max + 0.1, 1))

item_rows = len(item_names)
fig, axes = plt.subplots(item_rows, 2, figsize=(18, max(4 * item_rows, 24)))

if item_rows == 1:
    axes = [axes]

for row_axes, item_name in zip(axes, item_names):
    item_df = performance_df[performance_df['item_name'] == item_name]
    item_performance = item_df.groupby('participant_id').agg({
        'fluency': 'sum',
        'originality': 'mean',
    }).reset_index()

    sns.histplot(data=item_performance, x='fluency', kde=True, ax=row_axes[0], color='skyblue', edgecolor='black', binwidth=1)
    row_axes[0].xaxis.set_major_locator(ticker.MultipleLocator(1))
    row_axes[0].set_xlim(fluency_xlim)
    row_axes[0].set_title(f'{item_name} - 流畅性', fontsize=12)
    row_axes[0].set_xlabel('流畅性', fontsize=10)
    row_axes[0].set_ylabel('频数', fontsize=10)
    row_axes[0].grid(axis='y', linestyle='--', alpha=0.7)
    row_axes[0].set_axisbelow(True)

    originality_data = item_performance['originality'].dropna()
    sns.histplot(data=originality_data, kde=True, ax=row_axes[1], color='salmon', edgecolor='black', binwidth=0.1)
    row_axes[1].xaxis.set_major_locator(ticker.MultipleLocator(0.1))
    row_axes[1].set_xlim(originality_xlim)
    row_axes[1].set_title(f'{item_name} - 原创性', fontsize=12)
    row_axes[1].set_xlabel('原创性', fontsize=10)
    row_axes[1].set_ylabel('频数', fontsize=10)
    row_axes[1].grid(axis='y', linestyle='--', alpha=0.7)
    row_axes[1].set_axisbelow(True)

    sns.despine(ax=row_axes[0])
    sns.despine(ax=row_axes[1])

plt.tight_layout()
plt.show()